# FEVERScope: Full 52K Benchmark Pinecone Indexing (Google Colab)

This standalone Colab notebook runs on **GPU** to:
1. Encode the **52,051 sentences** from `corpus.parquet` with `sentence-transformers` (`all-MiniLM-L6-v2`, 384 dimensions).
2. Create a serverless **Pinecone** index.
3. Upsert dense sentence vectors with metadata (`doc_idx`, `doc_id`, `page_title`, `sent_id`, `text`).

> **GPU Recommended**: Go to `Runtime` -> `Change runtime type` -> select `T4 GPU`.

In [ ]:
# Step 1: Install required libraries
!pip install -q sentence-transformers pinecone-client pandas tqdm pyarrow

In [ ]:
# Step 2: Configure Pinecone Credentials
import os

PINECONE_API_KEY = "YOUR_PINECONE_API_KEY"       # <-- Replace with your Pinecone API key
PINECONE_INDEX_NAME = "fever-evidence-index"      # <-- Your chosen index name
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"         # 384 dimensions
VECTOR_DIMENSION = 384
BATCH_SIZE = 200                                  # Fast GPU batch size

In [ ]:
# Step 3: Initialize Pinecone client & Create Index
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

# Check existing indexes
existing_indexes = [idx["name"] for idx in pc.list_indexes()]
print(f"Existing indexes: {existing_indexes}")

if PINECONE_INDEX_NAME not in existing_indexes:
    print(f"Creating index: {PINECONE_INDEX_NAME} (dimension={VECTOR_DIMENSION}, metric=cosine)...")
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=VECTOR_DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print("Index created successfully!")
else:
    print(f"Index {PINECONE_INDEX_NAME} already exists. Ready to connect.")

index = pc.Index(PINECONE_INDEX_NAME)
print(index.describe_index_stats())

In [ ]:
# Step 4: Load Sentence Transformer on GPU
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)
print(f"Loaded {EMBEDDING_MODEL_NAME} on {device}.")

In [ ]:
# Step 5: Upload corpus.parquet (9.7 MB)
from google.colab import files
import pandas as pd

print("Upload corpus.parquet (from your Downloads folder):")
uploaded = files.upload()
corpus_filename = list(uploaded.keys())[0]
df = pd.read_parquet(corpus_filename)

print(f"Successfully loaded {len(df):,} sentences from {corpus_filename}!")
display(df.head(3))

In [ ]:
# Step 6: Encode and Upsert to Pinecone in Fast Batches (~2-3 mins on T4 GPU)
from tqdm.auto import tqdm

print(f"Embedding and upserting {len(df):,} sentences into Pinecone index '{PINECONE_INDEX_NAME}'...")

for start_idx in tqdm(range(0, len(df), BATCH_SIZE)):
    end_idx = min(start_idx + BATCH_SIZE, len(df))
    batch_df = df.iloc[start_idx:end_idx]
    
    # Prefer title-augmented text for richer context
    texts = (
        batch_df["text_with_title"].tolist()
        if "text_with_title" in batch_df.columns
        else batch_df["text"].tolist()
    )
    
    # Generate embeddings on GPU
    embeddings = model.encode(texts, batch_size=BATCH_SIZE, normalize_embeddings=True, show_progress_bar=False)
    
    # Format Pinecone vectors with rich metadata
    vectors_to_upsert = []
    for i, (_, row) in enumerate(batch_df.iterrows()):
        doc_id = str(row["doc_id"])
        doc_idx = int(row["doc_idx"]) if "doc_idx" in row else (start_idx + i)
        vector_values = embeddings[i].tolist()
        metadata = {
            "doc_id": doc_id,
            "doc_idx": doc_idx,
            "page_title": str(row["page_title"]),
            "sent_id": int(row["sent_id"]),
            "text": str(row["text"][:1000]),
        }
        vectors_to_upsert.append({"id": doc_id, "values": vector_values, "metadata": metadata})
        
    index.upsert(vectors=vectors_to_upsert)

print("All 52,051 sentences upserted to Pinecone successfully!")
print(index.describe_index_stats())

In [ ]:
# Step 7: Sanity Test Query
test_claim = "The Roman Empire was one of the largest empires in history."
query_embedding = model.encode(test_claim, normalize_embeddings=True).tolist()

results = index.query(
    vector=query_embedding,
    top_k=5,
    include_metadata=True
)

print(f"\nSearch Results for: '{test_claim}'")
for match in results["matches"]:
    score = match["score"]
    doc_id = match["id"]
    text = match["metadata"].get("text", "")
    print(f"- [{score:.4f}] {doc_id} -> {text[:120]}...")

## Step 8: Connect with FEVERScope Web App
1. Copy your `PINECONE_API_KEY`.
2. Open your FEVERScope React app (`http://localhost:5173`).
3. Click **⚙️ Pinecone Config** in the header, paste your key, and click **Save & Connect**.
The **Dense Semantic** column will immediately start querying this index!